# Figure 5: pathways

Panels a-d retain the original plotting cell. The optional SEM diagram is drawn from the saved result arrays, with variable-name checks for all regression paths and measurement loadings. Set `DRAW_SEM_DIAGRAM=False` in the final cell to export panels a-d only. Set `COMPUTE_SEM=True` in the first code cell for a new R/lavaan fit; the diagram then uses that fit. No manuscript image is required.


In [ ]:
import sys
from pathlib import Path

def find_repo(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config' / 'analysis.json').is_file():
            return candidate
    raise FileNotFoundError('Could not find config/analysis.json above the notebook directory')

ROOT = find_repo(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))
from elephant_analysis.runtime import configure_runtime
configure_runtime()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from elephant_analysis.artifacts import load_arrays, save_arrays
from elephant_analysis.data_inputs import load_inputs
from elephant_analysis.paired_sites import compute_paired
from elephant_analysis.partial_correlations import compute_figure5_correlations, correlation_results_to_arrays, figure5_plot_vectors
from elephant_analysis.sem import build_pathway_tables, load_sem_reference, pathway_tables_from_arrays, pathway_tables_to_arrays, run_sem, sem_result_to_arrays, verify_sem_reference_samples

DATA_ROOT = ROOT / 'data'
RESULTS_ROOT = ROOT / 'results'
FIGURES_ROOT = ROOT / 'figures'
INPUT_MODE = 'reference'  # Switch to raw to use the shared TIFF loader and paired core.
RUN_ANALYSIS = True  # False reads saved Fig. 5 results without loading inputs or invoking R.
RSCRIPT = None  # Optional full path to Rscript; otherwise use the environment/PATH.
COMPUTE_SEM = False  # True invokes R; the current reference fit has no recorded historical seed.
NEW_RUN_SEED = 20261107  # A fresh deterministic bootstrap seed, not the unknown historical seed.
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
if RUN_ANALYSIS:
    if INPUT_MODE == 'raw':
        raw_inputs = load_inputs('paired', DATA_ROOT, mode='raw')
        paired = compute_paired(raw_inputs)
    elif INPUT_MODE == 'reference':
        paired = load_inputs('paired', DATA_ROOT, mode='reference')

    else:
        raise ValueError('INPUT_MODE must be reference or raw')

    pathway_tables = build_pathway_tables(paired)
    pathway_arrays = pathway_tables_to_arrays(pathway_tables)
    pathway_cache = RESULTS_ROOT / 'fig5_pathway_tables.npz'
    save_arrays(pathway_cache, **pathway_arrays)
    cached_pathways = load_arrays(pathway_cache)
    pathway_tables = pathway_tables_from_arrays(cached_pathways)
    print('Fig. 5a-d table rows:', len(pathway_tables['ndvi']), len(pathway_tables['sif']))
    print('SEM complete cases:', len(pathway_tables['sem']))
    print('SEM columns:', list(pathway_tables['sem'].columns))

In [ ]:
figure5_result_path = RESULTS_ROOT / 'fig5_results.npz'
if RUN_ANALYSIS:
    correlations = compute_figure5_correlations(pathway_tables['ndvi'], pathway_tables['sif'])
    correlation_arrays = correlation_results_to_arrays(correlations)
    x_by_outcome, y_by_outcome = figure5_plot_vectors(pathway_tables['ndvi'], pathway_tables['sif'])
    plot_vector_arrays = {
        'plot_ndvi_x': np.vstack(x_by_outcome[0]),
        'plot_ndvi_y': np.vstack(y_by_outcome[0]),
        'plot_sif_x': np.vstack(x_by_outcome[1]),
        'plot_sif_y': np.vstack(y_by_outcome[1]),
    }

    if COMPUTE_SEM:
        sem_result = run_sem(
            pathway_tables['sem'],
            RESULTS_ROOT / 'sem_fit',
            script_path=ROOT / 'r' / 'fit_sem.R',
            rscript=RSCRIPT,
            n_boot=5000,
            seed=NEW_RUN_SEED,
        )
    else:
        sem_bundle = load_inputs('sem', DATA_ROOT, mode='reference')
        verify_sem_reference_samples(pathway_tables['sem'], sem_bundle)
        sem_result = load_sem_reference(sem_bundle)

    sem_arrays = sem_result_to_arrays(sem_result)
    figure5_results = dict(cached_pathways)
    figure5_results.update(correlation_arrays)
    figure5_results.update(plot_vector_arrays)
    figure5_results.update(sem_arrays)
    save_arrays(figure5_result_path, **figure5_results)
figure5_plot_data = load_arrays(figure5_result_path)
print('Partial-correlation rows:', len(figure5_plot_data['correlation_r']))
print('SEM software:', figure5_plot_data['sem_software'])

In [ ]:
# Original plotting cells copied from Main_figure.ipynb.
# Keep source sizes, font settings, labels, colours, positions and line widths.
EXPORT_PDF = True
EXPORT_DPI = 600
DRAW_SEM_DIAGRAM = True  # False exports Fig. 5a-d only.
SEM_SIZE_CM = (16, 17)
SEM_FONT_SIZE = 11
SEM_PATH_FONT_SIZE = 10
SEM_LINE_WIDTH = 1.4
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kstest, gaussian_kde
from elephant_analysis import array_utils as gc
mpl.rcdefaults()
mpl.rcParams["pdf.fonttype"] = 42

def export_original(figure, stem, dpi=EXPORT_DPI):
    # Match the source savefig contract, including tight bounding boxes.
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_ROOT / (stem + '.png'), dpi=dpi, bbox_inches='tight')
    if EXPORT_PDF:
        figure.savefig(FIGURES_ROOT / (stem + '.pdf'), bbox_inches='tight')

res_tree = [figure5_plot_data['plot_ndvi_x'], figure5_plot_data['plot_sif_x']]
res_res = [figure5_plot_data['plot_ndvi_y'][0], figure5_plot_data['plot_sif_y'][0]]
res_redius = [figure5_plot_data['plot_ndvi_y'][1], figure5_plot_data['plot_sif_y'][1]]
# BEGIN ORIGINAL CELL 47
import scipy
import matplotlib.pyplot as plt
import pingouin as pg
import seaborn as sns
import matplotlib.colors as colors
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(16,9))
fig.subplots_adjust(wspace=0.3,hspace=0.3, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)

cl=['royalblue','forestgreen']
mark=['*','.']
lab_x=['$Δ$TH (m)','$Δ$TH|$Δ$LST (m)','$Δ$TC (%)','$Δ$TC|$Δ$LST (%)']
lab_y=['$Δ$Resistance','$Δ$Resistance|$Δ$LST','$Δ$Resistance','$Δ$Resistance|$Δ$LST']
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        "trunc({n},{a:.2f},{b:.2f})".format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap
trunc_cmap = [truncate_colormap(plt.get_cmap("Blues"), 0.6, 1),truncate_colormap(plt.get_cmap("Greens"), 0.5, 0.9)]
cov=[[],'LST',[],'LST']
x_a=['TH','TH','TC','TC']
y_a=['NDVI_res','SIF_res']

for i in range(4):
    ax=fig.add_subplot(2,2,i+1)
    if i<=1:
        xx=np.linspace(-12,25,1000)
        ax.set_ylim(-0.2,0.2)
        ax.set_yticks(np.arange(-0.2,0.201,0.1))
        ax.set_xlim(-15,30)
        ax.set_xticks(np.arange(-15,30.1,15))
    else:
        xx=np.linspace(-40,60,1000)
        ax.set_ylim(-0.2,0.2)
        ax.set_yticks(np.arange(-0.2,0.201,0.1))
        ax.set_xlim(-50,75)
        ax.set_xticks(np.arange(-50,75.1,25))
    if i==0 or i==2:
        for j in range(2):
            kde_plot = sns.kdeplot(x=res_tree[j][i], y=res_res[j], cmap=trunc_cmap[j], alpha=0.5,fill=True,ax=ax,levels=30, thresh=0.04)
            poly=np.polyfit(x=res_tree[j][i], y=res_res[j],deg=1)
            y_value=np.polyval(poly,xx)
            print(i,f"slope={poly[0]}")

            cor = [round(float(figure5_plot_data['correlation_r'][2*i+j]), 3)]
            p_value = [float(figure5_plot_data['correlation_p'][2*i+j])]  # Apply thresholds before rounding.
            if p_value[0]<0.001:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.001'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.01:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.01'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.05:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.05'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]>=0.05:
                linestyle='--'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p={:.2f}'.format(cor[0],p_value[0]), transform = ax.transAxes,color=cl[j])
            ax.plot(xx,y_value,color=cl[j],ls=linestyle,linewidth=2)
    if i==1 or i==3:
        for j in range(2):
            kde_plot1 = sns.kdeplot(x=res_tree[j][i], y=res_redius[j], cmap=trunc_cmap[j], alpha=0.5,fill=True,ax=ax,levels=30, thresh=0.04)
            poly1=np.polyfit(x=res_tree[j][i], y=res_redius[j],deg=1)
            y_value1=np.polyval(poly1,xx)
            print(i,f"slope={poly1[0]}")

            cor = [round(float(figure5_plot_data['correlation_r'][2*i+j]), 3)]
            p_value = [float(figure5_plot_data['correlation_p'][2*i+j])]  # Apply thresholds before rounding.

            if p_value[0]<0.001:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.001'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.01:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.01'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.05:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.05'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]>=0.05:
                linestyle='--'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p={:.2f}'.format(cor[0],p_value[0]), transform = ax.transAxes,color=cl[j])
            ax.plot(xx,y_value1,color=cl[j],ls=linestyle,linewidth=2)
    ax.set_xlabel(lab_x[i],size=18,labelpad=10)
    ax.set_ylabel(lab_y[i],size=18,labelpad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
# END ORIGINAL CELL 47
export_original(fig, 'fig5a_d_partial_correlations')
plt.show()

# Draw the SEM from the currently loaded result arrays (saved fit or new R fit).
if DRAW_SEM_DIAGRAM:
    import json
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Ellipse
    from matplotlib.path import Path as MplPath
    from elephant_analysis.sem import sem_path_records

    sem_paths = sem_path_records(figure5_plot_data)
    # Coordinates and label locations are editable; coefficients are never typed here.
    sem_positions = {
        'Ele_Dens': (0.50, 0.92), 'tree': (0.50, 0.73),
        'TH': (0.15, 0.80), 'TC': (0.85, 0.80),
        'ET': (0.20, 0.53), 'Albedo': (0.80, 0.53),
        'LST': (0.37, 0.36), 'SPEI_m': (0.75, 0.29),
        'NDVI_res': (0.20, 0.09), 'SIF_res': (0.80, 0.09),
    }
    sem_labels = {
        'Ele_Dens': 'Elephant\ndensity', 'tree': 'Tree structure\n(latent)',
        'TH': 'ΔTH', 'TC': 'ΔTC', 'ET': 'ΔET', 'Albedo': 'ΔAlbedo',
        'LST': 'ΔLST', 'SPEI_m': 'Mean SPEI',
        'NDVI_res': 'ΔNDVI\nresistance', 'SIF_res': 'ΔSIF\nresistance',
    }
    # Cubic Bezier control points keep long direct paths away from intermediate nodes.
    # Dictionary key: (source, target); values: Bezier vertices followed by label position.
    sem_routes = {
        ('Ele_Dens', 'tree'): ((.50,.88),(.50,.86),(.50,.81),(.50,.78),(.59,.835)),
        ('tree','TH'): ((.42,.75),(.35,.78),(.29,.80),(.23,.80),(.31,.825)),
        ('tree','TC'): ((.58,.75),(.65,.78),(.71,.80),(.77,.80),(.69,.825)),
        ('tree','ET'): ((.44,.70),(.39,.65),(.31,.60),(.24,.57),(.34,.62)),
        ('tree','Albedo'): ((.56,.70),(.61,.65),(.69,.60),(.76,.57),(.65,.63)),
        ('tree','LST'): ((.48,.68),(.45,.60),(.39,.49),(.37,.40),(.40,.565)),
        ('ET','LST'): ((.23,.49),(.25,.45),(.30,.41),(.33,.395),(.21,.425)),
        ('Albedo','LST'): ((.73,.51),(.65,.47),(.52,.39),(.45,.37),(.61,.445)),
        ('tree','SPEI_m'): ((.58,.73),(.71,.72),(.91,.69),(.91,.60),(.91,.43),(.91,.29),(.83,.29),(.89,.385)),
        ('tree','NDVI_res'): ((.405,.73),(.23,.73),(.045,.70),(.045,.59),(.045,.33),(.045,.09),(.12,.09),(.08,.30)),
        ('tree','SIF_res'): ((.58,.745),(.80,.75),(.985,.72),(.985,.60),(.985,.28),(.985,.09),(.88,.09),(.94,.17)),
        ('LST','NDVI_res'): ((.34,.32),(.31,.25),(.24,.18),(.21,.13),(.24,.24)),
        ('LST','SIF_res'): ((.45,.35),(.54,.29),(.66,.16),(.73,.11),(.65,.175)),
        ('SPEI_m','NDVI_res'): ((.67,.28),(.56,.21),(.42,.10),(.28,.09),(.43,.125)),
        ('SPEI_m','SIF_res'): ((.76,.25),(.77,.21),(.79,.17),(.80,.13),(.83,.205)),
    }
    path_keys = {(row['source'], row['target']) for row in sem_paths}
    if path_keys != set(sem_routes):
        raise ValueError('SEM figure routes do not match the saved path identities')
    with mpl.rc_context({'font.family': 'Arial', 'font.size': SEM_FONT_SIZE,
                         'svg.fonttype': 'none', 'pdf.fonttype': 42, 'savefig.bbox': None}):
        sem_fig = plt.figure(figsize=(SEM_SIZE_CM[0]/2.54, SEM_SIZE_CM[1]/2.54))
        sem_ax = sem_fig.add_axes([.02, .17, .96, .78])
        sem_ax.set_xlim(0, 1)
        sem_ax.set_ylim(0, 1)
        sem_ax.axis('off')
        for row in sem_paths:
            *vertices, label_xy = sem_routes[(row['source'], row['target'])]
            significant = row['pvalue'] is not None and row['pvalue'] < .05
            linestyle = '-' if significant or row['fixed_unstandardized'] else '--'
            colour = '#355C7D' if row['std_all'] >= 0 else '#B8653B'
            curve = MplPath(vertices, [MplPath.MOVETO]+[MplPath.CURVE4]*(len(vertices)-1))
            sem_ax.add_patch(FancyArrowPatch(path=curve, arrowstyle='-|>', mutation_scale=11,
                                             linewidth=SEM_LINE_WIDTH, linestyle=linestyle, color=colour, zorder=1))
            if row['fixed_unstandardized']:
                suffix = '†'
            else:
                p = row['pvalue']
                suffix = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ' ns'
            row['display_label'] = f"{row['std_all']:.3f}{suffix}"
            sem_ax.text(*label_xy, row['display_label'], fontsize=SEM_PATH_FONT_SIZE,
                        ha='center', va='center', color=colour, zorder=4,
                        bbox={'facecolor':'white', 'edgecolor':'none', 'pad':1.2})
        for node, (x,y) in sem_positions.items():
            if node == 'tree':
                patch = Ellipse((x,y), .19, .10, facecolor='#EBF0F4', edgecolor='#4F5962', linewidth=1.2)
            else:
                patch = FancyBboxPatch((x-.08,y-.04), .16, .08, boxstyle='round,pad=0.004,rounding_size=0.012',
                                      facecolor='#F7F8F9', edgecolor='#6D747B', linewidth=1.0)
            patch.set_zorder(3)
            sem_ax.add_patch(patch)
            sem_ax.text(x, y, sem_labels[node], ha='center', va='center', fontsize=SEM_FONT_SIZE, zorder=5)
        sem_fig.text(.5, .975, 'SEM standardized paths', ha='center', va='top', fontsize=12)
        sem_fig.text(.04, .13, 'β = std.all; * p<0.05, ** p<0.01, *** p<0.001.', fontsize=10)
        sem_fig.text(.04, .095, 'Dashed: p≥0.05; † unstandardized loading fixed to 1.', fontsize=10)
        sem_fig.text(.04, .060, 'Regression paths and measurement loadings only.', fontsize=10)
        sem_fig.text(.04, .025, f"n = {len(figure5_plot_data['sem_values']):,}; residual covariances not shown.", fontsize=10)
        FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
        sem_fig.savefig(FIGURES_ROOT/'fig5_sem_paths.png', dpi=EXPORT_DPI, bbox_inches=None)
        if EXPORT_PDF:
            sem_fig.savefig(FIGURES_ROOT/'fig5_sem_paths.pdf', bbox_inches=None)
        plt.show()
    # Save the full-precision values underlying every displayed arrow, without CSV.
    path_record = {'coefficient_field':'sem_std_all', 'pvalue_field':'sem_pvalue',
                   'n':len(figure5_plot_data['sem_values']), 'paths':sem_paths,
                   'scope':'Regression paths and measurement loadings; residual covariances omitted.'}
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    (RESULTS_ROOT/'sem_paths.json').write_text(json.dumps(path_record,indent=2,allow_nan=False)+'\n', encoding='utf-8')
    display(pd.DataFrame(sem_paths)[['source','target','op','std_all','pvalue','fixed_unstandardized']])
